In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

import code_for_project_1
from importlib import reload

reload(code_for_project_1)
get_data = code_for_project_1.get_data
save_experiment_outputs = code_for_project_1.save_experiment_outputs

In [2]:
index_return_df, market_value_df = get_data(refresh_csv=True)
returns_df, market_df = index_return_df.copy(), market_value_df.copy()

D:\python_learning_virtual_evns\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
D:\python_learning_virtual_evns\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [3]:
returns_df
returns_df.columns, len(returns_df), returns_df['Date'].min(), returns_df['Date'].max()

(Index(['Date', 'Russell 1000 Value', 'Russell 1000 Growth',
        'Russell 2000 Value', 'Russell 2000 Growth',
        'MSCI World Ex USA Value NR USD', 'MSCI World Ex USA Growth NR USD',
        'MSCI World Ex USA Small Value NR USD',
        'MSCI World Ex USA Small Growth NR USD', 'MSCI EM Value NR USD',
        'MSCI EM Growth NR USD', 'MSCI EM Small Value NR USD',
        'MSCI EM Small Growth NR USD', 'Bloomberg Barclays US Aggregate',
        'Bloomberg Barclays Global Aggregate x USD',
        'Bloomberg Barclays Global Inflation-Linked USD',
        'Bloomberg Barclays US Municipal Bond',
        'Bloomberg Barclays Global High Yield USD', 'Bloomberg Commodity Index',
        'ICE BofA US Dollar Deposit Offered Rate Constant Maturity (1M)'],
       dtype='str'),
 300,
 np.int64(20010131),
 np.int64(20251231))

In [4]:
market_df
market_df.columns, len(market_df), market_df['Date'].min(), market_df['Date'].max()

(Index(['Date', 'Russell 1000 Value', 'Russell 1000 Growth',
        'Russell 2000 Value', 'Russell 2000 Growth',
        'MSCI World Ex USA Value NR USD', 'MSCI World Ex USA Growth NR USD',
        'MSCI World Ex USA Small Value NR USD',
        'MSCI World Ex USA Small Growth NR USD', 'MSCI EM Value NR USD',
        'MSCI EM Growth NR USD', 'MSCI EM Small Value NR USD',
        'MSCI EM Small Growth NR USD', 'Bloomberg Barclays US Aggregate',
        'Bloomberg Barclays Global Aggregate x USD',
        'Bloomberg Barclays Global Inflation-Linked',
        'Bloomberg Barclays Municipal Bond',
        'Bloomberg Barclays Global High Yield'],
       dtype='str'),
 223,
 np.int64(20070629),
 np.int64(20251231))

### Asset

In [5]:
#first test of whether there are some missing asset
for column in returns_df.columns:
    if column not in market_df.columns:
        print(f'{column} not in market_df')

Bloomberg Barclays Global Inflation-Linked USD not in market_df
Bloomberg Barclays US Municipal Bond not in market_df
Bloomberg Barclays Global High Yield USD not in market_df
Bloomberg Commodity Index not in market_df
ICE BofA US Dollar Deposit Offered Rate Constant Maturity (1M) not in market_df


In [6]:
# use the same column name in two csv
market_df.rename(columns={
    'Bloomberg Barclays Global Inflation-Linked': 'Bloomberg Barclays Global Inflation-Linked USD',
    'Bloomberg Barclays Municipal Bond': 'Bloomberg Barclays US Municipal Bond',
    'Bloomberg Barclays Global High Yield': 'Bloomberg Barclays Global High Yield USD'}, inplace=True)

In [7]:
#second test of different assets
mkt_lack = [column for column in returns_df.columns if column not in market_df.columns]
ret_lack = [column for column in market_df.columns if column not in returns_df.columns]
print('mkt_lack:', mkt_lack)
print('ret_lack:', ret_lack)

mkt_lack: ['Bloomberg Commodity Index', 'ICE BofA US Dollar Deposit Offered Rate Constant Maturity (1M)']
ret_lack: []


In [8]:
#freeze the common asset
COMMON_ASSETS = [column for column in returns_df.columns if column in market_df.columns and column != 'Date']
print(COMMON_ASSETS, len(COMMON_ASSETS))
returns_common = returns_df[['Date'] + COMMON_ASSETS].copy()
market_common = market_df[['Date'] + COMMON_ASSETS].copy()

['Russell 1000 Value', 'Russell 1000 Growth', 'Russell 2000 Value', 'Russell 2000 Growth', 'MSCI World Ex USA Value NR USD', 'MSCI World Ex USA Growth NR USD', 'MSCI World Ex USA Small Value NR USD', 'MSCI World Ex USA Small Growth NR USD', 'MSCI EM Value NR USD', 'MSCI EM Growth NR USD', 'MSCI EM Small Value NR USD', 'MSCI EM Small Growth NR USD', 'Bloomberg Barclays US Aggregate', 'Bloomberg Barclays Global Aggregate x USD', 'Bloomberg Barclays Global Inflation-Linked USD', 'Bloomberg Barclays US Municipal Bond', 'Bloomberg Barclays Global High Yield USD'] 17


### Time Check

In [9]:
#common missing value markers
COMMON_MISSING_VALUES = [
    '', ' ', 'NA', 'N/A', 'na', 'n/a',
    'NaN', 'nan', 'NULL', 'null',
    'None', 'none', '-', '--', '.'
]

In [10]:
#1.Standardize dates' types
returns_common['Date'] = returns_common['Date'].replace(COMMON_MISSING_VALUES, np.nan)
market_common['Date'] = market_common['Date'].replace(COMMON_MISSING_VALUES, np.nan)
returns_common['Date'] = pd.to_datetime(
    returns_common['Date'].astype('string'), format='%Y%m%d', errors='coerce'
)
market_common['Date'] = pd.to_datetime(
    market_common['Date'].astype('string'), format='%Y%m%d', errors='coerce'
)

#2. sort(from past to now)
returns_common = returns_common.sort_values('Date').reset_index(drop=True)
market_common = market_common.sort_values('Date').reset_index(drop=True)

#basic check
print('returns duplicated dates:',
      returns_common['Date'].duplicated().sum())
print('market duplicated dates:',
      market_common['Date'].duplicated().sum())

print('returns Date missing/invalid after cleaning:',
      returns_common['Date'].isna().sum())
print('market Date missing/invalid after cleaning:',
      market_common['Date'].isna().sum())

returns duplicated dates: 0
market duplicated dates: 0
returns Date missing/invalid after cleaning: 0
market Date missing/invalid after cleaning: 0


In [11]:
#3. check row-by-row year-month continuity
returns_month_num = (
        returns_common['Date'].dt.year * 12 + returns_common['Date'].dt.month)
market_month_num = (
        market_common['Date'].dt.year * 12 + market_common['Date'].dt.month)

returns_month_breaks = returns_common.loc[
    returns_month_num.diff().fillna(1).ne(1), 'Date']
market_month_breaks = market_common.loc[
    market_month_num.diff().fillna(1).ne(1), 'Date']

print('returns row-by-row year-month breaks:')
print(returns_month_breaks.tolist())
print('market row-by-row year-month breaks:')
print(market_month_breaks.tolist())

#4. check whether market dates are fully covered by returns months


market_date_not_in_returns = set(market_common['Date']) - set(returns_common['Date'])
returns_date_not_in_market = set(returns_common['Date']) - set(market_common['Date'])

print('market dates missing in returns:', len(market_date_not_in_returns))
print('returns dates not in market:', len(returns_date_not_in_market))


returns row-by-row year-month breaks:
[]
market row-by-row year-month breaks:
[]
market dates missing in returns: 0
returns dates not in market: 77


### Asset Check

In [12]:
for df in [returns_common, market_common]:
    df[COMMON_ASSETS] = (
        df[COMMON_ASSETS]
        .replace(COMMON_MISSING_VALUES, np.nan)
        .apply(pd.to_numeric, errors='coerce')
    )

#check all common asset columns
returns_missing_by_asset = (
    returns_common[COMMON_ASSETS].isna().sum().sort_values(ascending=False))
market_missing_by_asset = (
    market_common[COMMON_ASSETS].isna().sum().sort_values(ascending=False))

print('returns assets with missing:')
print(returns_missing_by_asset[returns_missing_by_asset > 0])
print('market assets with missing:')
print(market_missing_by_asset[market_missing_by_asset > 0])

returns assets with missing:
Series([], dtype: int64)
market assets with missing:
Series([], dtype: int64)


In [13]:
market_non_positive = (
    (market_common[COMMON_ASSETS] <= 0).sum().sort_values(ascending=False)
)
print('market non-positive by asset:')
print(market_non_positive[market_non_positive > 0])

market non-positive by asset:
Series([], dtype: int64)


In [14]:
returns_less_than_1 = (
    (returns_common[COMMON_ASSETS] <= -1).sum().sort_values(ascending=False)
)
print(
    f"returns less than 1 by asset: {returns_less_than_1[returns_less_than_1 > 0]}")

returns less than 1 by asset: Series([], dtype: int64)


### Construct Benchmark

In [15]:
FORMATION_DATES = 6

formation_candidates = (
    market_common.loc[
        market_common['Date'].dt.month.eq(FORMATION_DATES), ['Date']]
    .assign(Year=market_common['Date'].dt.year)
    .groupby('Year', as_index=False)['Date'].max()
    .rename(columns={'Date': 'formation_dates'})
)

formation_candidates

,Year,formation_dates
0,2007,2007-06-29
1,2008,2008-06-30
2,2009,2009-06-30
3,2010,2010-06-30
4,2011,2011-06-30
5,2012,2012-06-29
6,2013,2013-06-28
7,2014,2014-06-30
8,2015,2015-06-30
9,2016,2016-06-30


In [16]:
#we need 36 months before the date we construct a portfolio at least(returns) and 12 months after because we want to keep the portfolio for a whole year.

valid_formation_dates = []
non_valid_formation_dates = []
for formation_date in formation_candidates['formation_dates']:
    previous_months = (returns_common['Date'] < formation_date).sum()
    future_months = (returns_common['Date'] > formation_date).sum()

    if previous_months >= 60 and future_months >= 12:
        valid_formation_dates.append(formation_date)

    else:
        non_valid_formation_dates.append(formation_date)

print('formation dates valid:')
print(valid_formation_dates)
print('non-valid formation dates valid:')
print(non_valid_formation_dates)



formation dates valid:
[Timestamp('2007-06-29 00:00:00'), Timestamp('2008-06-30 00:00:00'), Timestamp('2009-06-30 00:00:00'), Timestamp('2010-06-30 00:00:00'), Timestamp('2011-06-30 00:00:00'), Timestamp('2012-06-29 00:00:00'), Timestamp('2013-06-28 00:00:00'), Timestamp('2014-06-30 00:00:00'), Timestamp('2015-06-30 00:00:00'), Timestamp('2016-06-30 00:00:00'), Timestamp('2017-06-30 00:00:00'), Timestamp('2018-06-29 00:00:00'), Timestamp('2019-06-28 00:00:00'), Timestamp('2020-06-30 00:00:00'), Timestamp('2021-06-30 00:00:00'), Timestamp('2022-06-30 00:00:00'), Timestamp('2023-06-30 00:00:00'), Timestamp('2024-06-28 00:00:00')]
non-valid formation dates valid:
[Timestamp('2025-06-30 00:00:00')]


In [17]:
benchmark_rows = []

for formation_date in valid_formation_dates:
    market_row = market_common.loc[
        market_common['Date'].eq(formation_date), COMMON_ASSETS
    ].iloc[0] #series

    weights = market_row / market_row.sum()
    benchmark_rows.append({'Date': formation_date, **weights.to_dict()})

benchmarks = pd.DataFrame(benchmark_rows)
benchmarks


,Date,Russell 1000 Value,Russell 1000 Growth,Russell 2000 Value,Russell 2000 Growth,MSCI World Ex USA Value NR USD,MSCI World Ex USA Growth NR USD,MSCI World Ex USA Small Value NR USD,MSCI World Ex USA Small Growth NR USD,MSCI EM Value NR USD,MSCI EM Growth NR USD,MSCI EM Small Value NR USD,MSCI EM Small Growth NR USD,Bloomberg Barclays US Aggregate,Bloomberg Barclays Global Aggregate x USD,Bloomberg Barclays Global Inflation-Linked USD,Bloomberg Barclays US Municipal Bond,Bloomberg Barclays Global High Yield USD
0,2007-06-29,0.120977,0.120805,0.011307,0.011415,0.117192,0.117593,0.013721,0.014220,0.022105,0.022105,0.003575,0.003728,0.147376,0.224747,0.016816,0.016270,0.016049
1,2008-06-30,0.097725,0.105320,0.008113,0.008371,0.102382,0.108045,0.010282,0.011693,0.024820,0.024416,0.003175,0.003147,0.168959,0.269946,0.021939,0.017314,0.014355
2,2009-06-30,0.077594,0.081054,0.006450,0.006673,0.080207,0.079172,0.008403,0.008764,0.020946,0.020907,0.002948,0.002919,0.216049,0.329225,0.023145,0.020294,0.015251
3,2010-06-30,0.082590,0.081065,0.007042,0.007348,0.074385,0.077409,0.010097,0.010023,0.022950,0.022697,0.003332,0.003212,0.227160,0.311298,0.021587,0.019218,0.018587
4,2011-06-30,0.089483,0.087743,0.008003,0.008109,0.082261,0.082554,0.011418,0.011318,0.024271,0.025673,0.003466,0.003413,0.195538,0.309358,0.022134,0.015792,0.019467
5,2012-06-29,0.091766,0.090019,0.007524,0.007526,0.070276,0.069363,0.009504,0.009604,0.021272,0.021938,0.002851,0.002794,0.216307,0.316170,0.025358,0.017404,0.020324
6,2013-06-28,0.104404,0.095965,0.008639,0.008274,0.076097,0.076589,0.010304,0.010486,0.020120,0.021410,0.003034,0.002987,0.204002,0.294627,0.024449,0.015892,0.022721
7,2014-06-30,0.106724,0.108047,0.009371,0.009367,0.080590,0.080434,0.011927,0.011961,0.020117,0.021670,0.003225,0.003265,0.183415,0.286629,0.025326,0.014078,0.023854
8,2015-06-30,0.113777,0.115948,0.010221,0.010441,0.077142,0.077990,0.011616,0.012009,0.021636,0.021600,0.003498,0.003638,0.196213,0.258426,0.026391,0.014795,0.024658
9,2016-06-30,0.109040,0.108193,0.008579,0.008703,0.067011,0.069049,0.010898,0.011165,0.019885,0.019817,0.003190,0.003240,0.206781,0.286585,0.027555,0.015516,0.024793


In [18]:
expected_columns = ['Date'] + COMMON_ASSETS
row_sums = benchmarks[COMMON_ASSETS].sum(axis=1)

print('benchmark rows:', len(benchmarks))
print('expected rows:', len(valid_formation_dates))
print('duplicate dates:', benchmarks['Date'].duplicated().sum())
print('missing values:', benchmarks[COMMON_ASSETS].isna().sum().sum())
print('min weight:', benchmarks[COMMON_ASSETS].min().min())
print('max weight:', benchmarks[COMMON_ASSETS].max().max())
print('max |row sum - 1|:', (row_sums - 1).abs().max())

assert benchmarks.columns.tolist() == expected_columns
assert len(benchmarks) == len(valid_formation_dates)
assert benchmarks['Date'].tolist() == list(valid_formation_dates)
assert benchmarks['Date'].is_unique
assert benchmarks[COMMON_ASSETS].notna().all().all()
assert np.isfinite(benchmarks[COMMON_ASSETS].to_numpy()).all()
assert (benchmarks[COMMON_ASSETS] >= 0).all().all()
assert np.allclose(row_sums.to_numpy(), 1.0)

print('benchmark checks passed')


benchmark rows: 18
expected rows: 18
duplicate dates: 0
missing values: 0
min weight: 0.002625479736661791
max weight: 0.3292248010316403
max |row sum - 1|: 2.220446049250313e-16
benchmark checks passed


### BL Calculation

In [ ]:
# covariance comparison setup
COVARIANCE_WINDOW = 60
COVARIANCE_METHODS = [
    'sample_covariance_no_shrinkage',
    'sklearn_ledoit_wolf',
]
PRIMARY_COVARIANCE_METHOD = 'sample_covariance_no_shrinkage'


def estimate_covariance_matrices(method):
    cov_matrices_month = {}
    cov_matrices_ann = {}
    shrinkage_by_date = {}
    diagnostic_rows = []

    for formation_date in valid_formation_dates:
        return_window = returns_common.loc[
            returns_common['Date'] < formation_date, ['Date'] + COMMON_ASSETS
        ].tail(COVARIANCE_WINDOW)

        if len(return_window) != COVARIANCE_WINDOW:
            raise ValueError(
                f'{formation_date}: expected {COVARIANCE_WINDOW} months, got {len(return_window)}'
            )

        returns_window = return_window[COMMON_ASSETS].astype(float)

        if method == 'sample_covariance_no_shrinkage':
            cov_month = returns_window.cov()
            shrinkage = np.nan
        elif method == 'sklearn_ledoit_wolf':
            estimator = LedoitWolf(assume_centered=False).fit(returns_window.to_numpy())
            cov_month = pd.DataFrame(
                estimator.covariance_,
                index=COMMON_ASSETS,
                columns=COMMON_ASSETS,
            )
            shrinkage = float(estimator.shrinkage_)
            shrinkage_by_date[formation_date] = shrinkage
        else:
            raise ValueError(f'Unknown covariance method: {method}')

        cov_ann = 12 * cov_month
        eigvals = np.linalg.eigvalsh(cov_ann.to_numpy())
        min_eig = float(eigvals.min())
        max_eig = float(eigvals.max())
        condition_number = np.inf if min_eig <= 0 else max_eig / min_eig

        cov_matrices_month[formation_date] = cov_month
        cov_matrices_ann[formation_date] = cov_ann
        diagnostic_rows.append({
            'Date': formation_date,
            'Method': method,
            'Min Eigenvalue': min_eig,
            'Max Eigenvalue': max_eig,
            'Condition Number': condition_number,
            'Ledoit-Wolf Shrinkage': shrinkage,
        })

    return {
        'method': method,
        'cov_matrices_month': cov_matrices_month,
        'cov_matrices_ann': cov_matrices_ann,
        'cov_shrinkage_by_date': pd.Series(shrinkage_by_date, name='Ledoit-Wolf Shrinkage'),
        'covariance_diagnostics': pd.DataFrame(diagnostic_rows),
    }


covariance_results = {
    method: estimate_covariance_matrices(method)
    for method in COVARIANCE_METHODS
}

covariance_diagnostics = pd.concat(
    [result['covariance_diagnostics'] for result in covariance_results.values()],
    ignore_index=True,
)

covariance_diagnostics_summary = (
    covariance_diagnostics
    .groupby('Method')
    .agg(
        formation_dates=('Date', 'count'),
        mean_condition_number=('Condition Number', 'mean'),
        max_condition_number=('Condition Number', 'max'),
        min_eigenvalue=('Min Eigenvalue', 'min'),
        mean_ledoit_wolf_shrinkage=('Ledoit-Wolf Shrinkage', 'mean'),
    )
    .reset_index()
)

selected_covariance_result = covariance_results[PRIMARY_COVARIANCE_METHOD]
COVARIANCE_METHOD = selected_covariance_result['method']
cov_matrices_month = selected_covariance_result['cov_matrices_month']
cov_matrices_ann = selected_covariance_result['cov_matrices_ann']
cov_shrinkage_by_date = selected_covariance_result['cov_shrinkage_by_date']

display(covariance_diagnostics_summary)
cov_matrices_ann[valid_formation_dates[0]]

In [ ]:
# model parameters shared by every covariance-method comparison
lambda_ = 2.5
active_bound = 0.02

tau = 0.05
LOOKBACK_SIGNAL_MONTHS = 12

### View Design

We keep two **weak, benchmark-relative views** and treat them as modest tilts rather than aggressive forecasts.

1. **Bloomberg Barclays Global Inflation-Linked USD - Bloomberg Barclays Global Aggregate x USD.** This is a relative inflation-sensitivity tilt inside the bond sleeve. The long leg keeps some inflation protection, while the short leg stays in the broad nominal global aggregate benchmark.
2. **MSCI EM Value NR USD - MSCI EM Growth NR USD.** This adds a non-bond tilt so the final portfolio is not expressing all of its active views only inside fixed income. The tilt is still weak and benchmark-relative: it prefers EM value over EM growth, but only when the recent evidence supports keeping that view alive.

To avoid look-ahead bias, the **view directions stay fixed**, but `Q_t` and `Omega_t` are now **scaled by the trailing 12-month relative spread observed before each formation date**. Stronger recent evidence keeps more of the base view; weaker or negative evidence shrinks `Q_t` and increases `Omega_t`, which is more consistent with the assignment's weak-confidence requirement than aggressively flipping the view direction.


In [ ]:
view_specs = pd.DataFrame(
    [
        {
            'View': 'Bloomberg Barclays Global Inflation-Linked USD - Bloomberg Barclays Global Aggregate x USD',
            'Long Asset': 'Bloomberg Barclays Global Inflation-Linked USD',
            'Short Asset': 'Bloomberg Barclays Global Aggregate x USD',
            'Q_annual': 0.0015,
            'Omega_multiplier': 8.0,
            'Comment': 'Weak inflation-protection tilt inside the bond sleeve.',
        },
        {
            'View': 'MSCI EM Value NR USD - MSCI EM Growth NR USD',
            'Long Asset': 'MSCI EM Value NR USD',
            'Short Asset': 'MSCI EM Growth NR USD',
            'Q_annual': 0.0005,
            'Omega_multiplier': 12.0,
            'Comment': 'Very weak EM value-over-growth tilt.',
        },
    ]
).set_index('View')

P = pd.DataFrame(0.0, index=view_specs.index, columns=COMMON_ASSETS)

for view_name, spec in view_specs.iterrows():
    P.loc[view_name, spec['Long Asset']] = 1.0
    P.loc[view_name, spec['Short Asset']] = -1.0


def scale_view_strength(trailing_spread_ann):
    if trailing_spread_ann >= 0.02:
        return 1.0, 1.0
    if trailing_spread_ann >= 0.0:
        return 0.5, 2.0
    if trailing_spread_ann >= -0.02:
        return 0.25, 4.0
    return 0.10, 8.0


print('P shape:', P.shape)
display(view_specs)
display(P)

In [ ]:
def build_implied_returns(cov_matrices_ann):
    implied_returns = {}

    for formation_date in valid_formation_dates:
        benchmark_weights = benchmarks.loc[
            benchmarks['Date'].eq(formation_date), COMMON_ASSETS
        ].iloc[0]
        sigma_ann = cov_matrices_ann[formation_date]
        benchmark_weights = benchmark_weights.reindex(sigma_ann.index)
        implied_returns[formation_date] = lambda_ * sigma_ann @ benchmark_weights

    return implied_returns


def build_view_inputs(cov_matrices_ann):
    Q_by_date = {}
    Omega_by_date = {}
    Omega_multiplier_by_date = {}
    adaptive_view_rows = []

    for formation_date in valid_formation_dates:
        sigma_ann = cov_matrices_ann[formation_date].loc[COMMON_ASSETS, COMMON_ASSETS]
        q = view_specs['Q_annual'].copy()
        omega_multiplier = view_specs['Omega_multiplier'].copy()

        for view_name, spec in view_specs.iterrows():
            trailing_window = returns_common.loc[
                returns_common['Date'] < formation_date,
                [spec['Long Asset'], spec['Short Asset']],
            ].tail(LOOKBACK_SIGNAL_MONTHS)

            if len(trailing_window) != LOOKBACK_SIGNAL_MONTHS:
                raise ValueError(
                    f'{formation_date}: expected {LOOKBACK_SIGNAL_MONTHS} signal months, got {len(trailing_window)}'
                )

            trailing_spread_ann = float(
                12 * (trailing_window[spec['Long Asset']] - trailing_window[spec['Short Asset']]).mean()
            )
            q_scale, omega_scale = scale_view_strength(trailing_spread_ann)

            q.loc[view_name] = q.loc[view_name] * q_scale
            omega_multiplier.loc[view_name] = omega_multiplier.loc[view_name] * omega_scale
            adaptive_view_rows.append({
                'Date': formation_date,
                'View': view_name,
                'Trailing Spread Annualized': trailing_spread_ann,
                'Q Scale': q_scale,
                'Omega Scale': omega_scale,
                'Q_annual_t': q.loc[view_name],
                'Omega_multiplier_t': omega_multiplier.loc[view_name],
            })

        q.name = formation_date
        projected_view_cov = P @ (tau * sigma_ann) @ P.T
        omega_diag = omega_multiplier.to_numpy() * np.diag(projected_view_cov.to_numpy())
        omega = pd.DataFrame(np.diag(omega_diag), index=P.index, columns=P.index)

        Q_by_date[formation_date] = q
        Omega_by_date[formation_date] = omega
        Omega_multiplier_by_date[formation_date] = omega_multiplier

    Q_table = pd.DataFrame(Q_by_date).T
    Q_table.index.name = 'Date'
    Omega_multiplier_table = pd.DataFrame(Omega_multiplier_by_date).T
    Omega_multiplier_table.index.name = 'Date'
    adaptive_view_summary = pd.DataFrame(adaptive_view_rows)

    return Q_by_date, Omega_by_date, Omega_multiplier_by_date, Q_table, Omega_multiplier_table, adaptive_view_summary

def build_bl_posterior(cov_matrices_ann, implied_returns, Q_by_date, Omega_by_date):
    bl_posterior_returns = {}
    bl_shift_from_prior = {}

    for formation_date in valid_formation_dates:
        sigma_ann = cov_matrices_ann[formation_date].loc[COMMON_ASSETS, COMMON_ASSETS]
        pi = implied_returns[formation_date].reindex(COMMON_ASSETS)
        q = Q_by_date[formation_date].reindex(P.index)
        omega = Omega_by_date[formation_date].loc[P.index, P.index]

        view_gap = q - P @ pi
        middle = P @ (tau * sigma_ann) @ P.T + omega
        solve_term = pd.Series(
            np.linalg.solve(middle.to_numpy(), view_gap.to_numpy()),
            index=P.index,
        )

        adjustment = tau * sigma_ann @ P.T @ solve_term
        bl_posterior_returns[formation_date] = pi + adjustment
        bl_shift_from_prior[formation_date] = adjustment

    bl_posterior_table = pd.DataFrame(bl_posterior_returns).T
    bl_posterior_table.index.name = 'Date'
    bl_shift_table = pd.DataFrame(bl_shift_from_prior).T
    bl_shift_table.index.name = 'Date'

    posterior_shift_summary = pd.DataFrame({
        'Max Abs Shift': bl_shift_table.abs().max(axis=1),
        'Mean Abs Shift': bl_shift_table.abs().mean(axis=1),
    })

    return bl_posterior_returns, bl_shift_from_prior, bl_posterior_table, bl_shift_table, posterior_shift_summary


def solve_optimized_weights(cov_matrices_ann, bl_posterior_returns):
    optimal_weight_rows = []
    optimization_diagnostics = []

    for formation_date in valid_formation_dates:
        sigma_ann = cov_matrices_ann[formation_date].loc[COMMON_ASSETS, COMMON_ASSETS]
        mu_bl = bl_posterior_returns[formation_date].reindex(COMMON_ASSETS)
        benchmark_weights = benchmarks.loc[
            benchmarks['Date'].eq(formation_date), COMMON_ASSETS
        ].iloc[0].reindex(COMMON_ASSETS)

        sigma_np = sigma_ann.to_numpy()
        mu_np = mu_bl.to_numpy()
        benchmark_np = benchmark_weights.to_numpy()
        bounds = [
            (max(0.0, w_b - active_bound), min(1.0, w_b + active_bound))
            for w_b in benchmark_np
        ]

        def objective(weights, mu=mu_np, sigma=sigma_np, risk_aversion=lambda_):
            return 0.5 * risk_aversion * weights @ sigma @ weights - mu @ weights

        result = minimize(
            objective,
            x0=benchmark_np,
            method='SLSQP',
            bounds=bounds,
            constraints=[{'type': 'eq', 'fun': lambda weights: weights.sum() - 1.0}],
            options={'maxiter': 500, 'ftol': 1e-12},
        )

        if not result.success:
            raise RuntimeError(f'{formation_date}: optimization failed - {result.message}')

        weights = pd.Series(result.x, index=COMMON_ASSETS)
        active_weights = weights - benchmark_weights
        optimal_weight_rows.append({'Date': formation_date, **weights.to_dict()})
        optimization_diagnostics.append({
            'Date': formation_date,
            'Objective Value': result.fun,
            'Weight Sum': weights.sum(),
            'Min Weight': weights.min(),
            'Max Weight': weights.max(),
            'Max Abs Active': active_weights.abs().max(),
            'Hit Active Bound Count': int((active_weights.abs() >= active_bound - 1e-6).sum()),
        })
    optimal_weights = pd.DataFrame(optimal_weight_rows)
    optimization_diagnostics = pd.DataFrame(optimization_diagnostics)
    active_weights_table = optimal_weights.set_index('Date')[COMMON_ASSETS].sub(
        benchmarks.set_index('Date')[COMMON_ASSETS],
        fill_value=0.0,
    )

    assert np.allclose(optimal_weights[COMMON_ASSETS].sum(axis=1).to_numpy(), 1.0)
    assert (optimal_weights[COMMON_ASSETS] >= -1e-10).all().all()
    assert (active_weights_table.abs() <= active_bound + 1e-6).all().all()

    return optimal_weights, active_weights_table, optimization_diagnostics


def run_backtest(optimal_weights):
    candidate_dates = formation_candidates['formation_dates'].tolist()
    portfolio_monthly_rows = []
    holding_period_rows = []

    for formation_date in valid_formation_dates:
        formation_idx = candidate_dates.index(formation_date)
        if formation_idx + 1 < len(candidate_dates):
            period_end = candidate_dates[formation_idx + 1]
        else:
            period_end = returns_common['Date'].max()

        holding_window = returns_common.loc[
            (returns_common['Date'] > formation_date) &
            (returns_common['Date'] <= period_end),
            ['Date'] + COMMON_ASSETS,
        ].copy()

        weights = optimal_weights.loc[
            optimal_weights['Date'].eq(formation_date), COMMON_ASSETS
        ].iloc[0].reindex(COMMON_ASSETS).astype(float)
        start_weights = weights.copy()

        for _, row in holding_window.iterrows():
            asset_returns = row[COMMON_ASSETS].astype(float)
            portfolio_return = float(weights @ asset_returns)
            portfolio_monthly_rows.append({
                'Date': row['Date'],
                'Formation Date': formation_date,
                'Portfolio Return': portfolio_return,
            })

            weights = weights * (1.0 + asset_returns)
            weights = weights / weights.sum()

        holding_period_rows.append({
            'Formation Date': formation_date,
            'Period End': period_end,
            'Months Held': int(len(holding_window)),
            'Start Weight Sum': float(start_weights.sum()),
            'End Weight Sum': float(weights.sum()),
        })

    portfolio_monthly_returns = pd.DataFrame(portfolio_monthly_rows)
    holding_period_summary = pd.DataFrame(holding_period_rows)
    assert (holding_period_summary['Months Held'] == 12).all()

    benchmark_monthly_weights = market_common[['Date'] + COMMON_ASSETS].copy()
    benchmark_monthly_weights[COMMON_ASSETS] = benchmark_monthly_weights[COMMON_ASSETS].div(
        benchmark_monthly_weights[COMMON_ASSETS].sum(axis=1),
        axis=0,
    )
    benchmark_weight_lag = benchmark_monthly_weights.set_index('Date')[COMMON_ASSETS].shift(1)
    realized_returns = returns_common.set_index('Date')[COMMON_ASSETS]
    benchmark_monthly_returns = (benchmark_weight_lag * realized_returns).sum(axis=1).rename(
        'Benchmark Return'
    )

    backtest_results = portfolio_monthly_returns.merge(
        benchmark_monthly_returns.reset_index(),
        on='Date',
        how='left',
    )
    backtest_results['Active Return'] = (
        backtest_results['Portfolio Return'] - backtest_results['Benchmark Return']
    )
    backtest_results['Portfolio Growth'] = (1.0 + backtest_results['Portfolio Return']).cumprod()
    backtest_results['Benchmark Growth'] = (1.0 + backtest_results['Benchmark Return']).cumprod()

    return backtest_results, holding_period_summary

def summarize_strategy(backtest_results, optimization_diagnostics, active_weights_table, cov_result):
    n_months = len(backtest_results)
    portfolio_ann_return = (backtest_results['Portfolio Growth'].iloc[-1] ** (12 / n_months)) - 1
    benchmark_ann_return = (backtest_results['Benchmark Growth'].iloc[-1] ** (12 / n_months)) - 1
    portfolio_ann_vol = backtest_results['Portfolio Return'].std(ddof=1) * np.sqrt(12)
    benchmark_ann_vol = backtest_results['Benchmark Return'].std(ddof=1) * np.sqrt(12)
    active_ann_return = backtest_results['Active Return'].mean() * 12
    tracking_error = backtest_results['Active Return'].std(ddof=1) * np.sqrt(12)
    information_ratio = np.nan if np.isclose(tracking_error, 0.0) else active_ann_return / tracking_error
    relative_final = backtest_results['Portfolio Growth'].iloc[-1] / backtest_results['Benchmark Growth'].iloc[-1] - 1.0

    cov_diag = cov_result['covariance_diagnostics']
    return {
        'portfolio_ann_return': float(portfolio_ann_return),
        'benchmark_ann_return': float(benchmark_ann_return),
        'portfolio_ann_vol': float(portfolio_ann_vol),
        'benchmark_ann_vol': float(benchmark_ann_vol),
        'active_ann_return': float(active_ann_return),
        'tracking_error': float(tracking_error),
        'information_ratio': np.nan if pd.isna(information_ratio) else float(information_ratio),
        'portfolio_final_growth': float(backtest_results['Portfolio Growth'].iloc[-1]),
        'benchmark_final_growth': float(backtest_results['Benchmark Growth'].iloc[-1]),
        'relative_final': float(relative_final),
        'max_abs_active': float(active_weights_table.abs().max().max()),
        'mean_abs_active': float(active_weights_table.abs().stack().mean()),
        'hit_active_bound_total': int(optimization_diagnostics['Hit Active Bound Count'].sum()),
        'years_with_any_hit_bound': int((optimization_diagnostics['Hit Active Bound Count'] > 0).sum()),
        'max_cov_condition_number': float(cov_diag['Condition Number'].max()),
        'mean_cov_condition_number': float(cov_diag['Condition Number'].mean()),
    }


def run_strategy_with_covariance(cov_result):
    cov_matrices_ann = cov_result['cov_matrices_ann']
    implied_returns = build_implied_returns(cov_matrices_ann)
    (
        Q_by_date,
        Omega_by_date,
        Omega_multiplier_by_date,
        Q_table,
        Omega_multiplier_table,
        adaptive_view_summary,
    ) = build_view_inputs(cov_matrices_ann)
    (
        bl_posterior_returns,
        bl_shift_from_prior,
        bl_posterior_table,
        bl_shift_table,
        posterior_shift_summary,
    ) = build_bl_posterior(cov_matrices_ann, implied_returns, Q_by_date, Omega_by_date)
    optimal_weights, active_weights_table, optimization_diagnostics = solve_optimized_weights(
        cov_matrices_ann,
        bl_posterior_returns,
    )
    backtest_results, holding_period_summary = run_backtest(optimal_weights)
    metrics = summarize_strategy(
        backtest_results,
        optimization_diagnostics,
        active_weights_table,
        cov_result,
    )

    return {
        **cov_result,
        'implied_returns': implied_returns,
        'Q_by_date': Q_by_date,
        'Omega_by_date': Omega_by_date,
        'Omega_multiplier_by_date': Omega_multiplier_by_date,
        'Q_table': Q_table,
        'Omega_multiplier_table': Omega_multiplier_table,
        'adaptive_view_summary': adaptive_view_summary,
        'bl_posterior_returns': bl_posterior_returns,
        'bl_shift_from_prior': bl_shift_from_prior,
        'bl_posterior_table': bl_posterior_table,
        'bl_shift_table': bl_shift_table,
        'posterior_shift_summary': posterior_shift_summary,
        'optimal_weights': optimal_weights,
        'active_weights_table': active_weights_table,
        'optimization_diagnostics': optimization_diagnostics,
        'backtest_results': backtest_results,
        'holding_period_summary': holding_period_summary,
        'metrics': metrics,
    }

In [ ]:
strategy_results = {
    method: run_strategy_with_covariance(covariance_results[method])
    for method in COVARIANCE_METHODS
}

comparison_summary = pd.DataFrame(
    [
        {
            'Method': method,
            'Portfolio Annual Return': result['metrics']['portfolio_ann_return'],
            'Portfolio Annual Volatility': result['metrics']['portfolio_ann_vol'],
            'Active Return': result['metrics']['active_ann_return'],
            'Tracking Error': result['metrics']['tracking_error'],
            'Information Ratio': result['metrics']['information_ratio'],
            'Final Relative Outperformance': result['metrics']['relative_final'],
            'Hit Active Bound Total': result['metrics']['hit_active_bound_total'],
            'Years With Any Active Bound Hit': result['metrics']['years_with_any_hit_bound'],
            'Max Covariance Condition Number': result['metrics']['max_cov_condition_number'],
        }
        for method, result in strategy_results.items()
    ]
)

comparison_summary_formatted = comparison_summary.copy()
percent_columns = [
    'Portfolio Annual Return',
    'Portfolio Annual Volatility',
    'Active Return',
    'Tracking Error',
    'Final Relative Outperformance',
]
for column in percent_columns:
    comparison_summary_formatted[column] = comparison_summary_formatted[column].map(lambda value: f'{value:.3%}')
comparison_summary_formatted['Information Ratio'] = comparison_summary_formatted['Information Ratio'].map(lambda value: f'{value:.3f}')
comparison_summary_formatted['Max Covariance Condition Number'] = comparison_summary_formatted['Max Covariance Condition Number'].map(lambda value: f'{value:.1f}')

display(comparison_summary_formatted)
display(covariance_diagnostics_summary)

### Backtest

I rebalance the active portfolio only at the annual formation dates. Between rebalances, weights are allowed to drift with realized returns. For the benchmark, monthly realized returns are computed using lagged monthly market-value weights times current-month asset returns.


In [ ]:
# Select one method for the existing single-strategy tables and optional export.
primary_result = strategy_results[PRIMARY_COVARIANCE_METHOD]
COVARIANCE_METHOD = primary_result['method']
cov_matrices_month = primary_result['cov_matrices_month']
cov_matrices_ann = primary_result['cov_matrices_ann']
cov_shrinkage_by_date = primary_result['cov_shrinkage_by_date']

implied_returns = primary_result['implied_returns']
Q_by_date = primary_result['Q_by_date']
Omega_by_date = primary_result['Omega_by_date']
Omega_multiplier_by_date = primary_result['Omega_multiplier_by_date']
Q_table = primary_result['Q_table']
Omega_multiplier_table = primary_result['Omega_multiplier_table']
adaptive_view_summary = primary_result['adaptive_view_summary']

bl_posterior_returns = primary_result['bl_posterior_returns']
bl_shift_from_prior = primary_result['bl_shift_from_prior']
bl_posterior_table = primary_result['bl_posterior_table']
bl_shift_table = primary_result['bl_shift_table']
posterior_shift_summary = primary_result['posterior_shift_summary']

optimal_weights = primary_result['optimal_weights']
active_weights_table = primary_result['active_weights_table']
optimization_diagnostics = primary_result['optimization_diagnostics']
backtest_results = primary_result['backtest_results']
holding_period_summary = primary_result['holding_period_summary']

first_date = valid_formation_dates[0]
comparison_first_date = pd.DataFrame({
    'Implied Return': implied_returns[first_date].reindex(COMMON_ASSETS),
    'BL Posterior Return': bl_posterior_returns[first_date].reindex(COMMON_ASSETS),
    'Posterior - Implied': bl_shift_from_prior[first_date].reindex(COMMON_ASSETS),
})

display(posterior_shift_summary)
display(comparison_first_date)
display(optimization_diagnostics)
display(holding_period_summary)
backtest_results.head()

In [ ]:
def make_performance_outputs(backtest_results, covariance_method, show=True):
    n_months = len(backtest_results)

    portfolio_ann_return = (backtest_results['Portfolio Growth'].iloc[-1] ** (12 / n_months)) - 1
    benchmark_ann_return = (backtest_results['Benchmark Growth'].iloc[-1] ** (12 / n_months)) - 1
    portfolio_ann_vol = backtest_results['Portfolio Return'].std(ddof=1) * np.sqrt(12)
    benchmark_ann_vol = backtest_results['Benchmark Return'].std(ddof=1) * np.sqrt(12)
    active_ann_return = backtest_results['Active Return'].mean() * 12
    tracking_error = backtest_results['Active Return'].std(ddof=1) * np.sqrt(12)
    information_ratio = np.nan if np.isclose(tracking_error, 0.0) else active_ann_return / tracking_error

    plt.style.use('default')
    series = backtest_results[['Date', 'Portfolio Growth', 'Benchmark Growth']].copy()
    series['Relative Performance'] = (
        series['Portfolio Growth'] / series['Benchmark Growth'] - 1.0
    )
    portfolio_color = '#111111'
    benchmark_color = '#8a8a8a'
    relative_color = '#b55d1f'
    grid_color = '#d7d7d7'

    summary_rows = [
        ['Covariance method', covariance_method],
        ['Portfolio annual return', f'{portfolio_ann_return:.1%}'],
        ['Portfolio annual volatility', f'{portfolio_ann_vol:.1%}'],
        ['Benchmark annual return', f'{benchmark_ann_return:.1%}'],
        ['Benchmark annual volatility', f'{benchmark_ann_vol:.1%}'],
        ['Active return', f'{active_ann_return:.1%}'],
        ['Tracking error', f'{tracking_error:.1%}'],
        ['Information ratio', 'N/A' if pd.isna(information_ratio) else f'{information_ratio:.2f}'],
    ]
    summary_df = pd.DataFrame(summary_rows, columns=['Metric', 'Value'])

    growth_fig, growth_ax = plt.subplots(
        figsize=(9.5, 5.2),
        facecolor='white',
        constrained_layout=True,
    )
    growth_ax.plot(series['Date'], series['Portfolio Growth'], color=portfolio_color, linewidth=2.3, label='Portfolio')
    growth_ax.plot(
        series['Date'],
        series['Benchmark Growth'],
        color=benchmark_color,
        linewidth=2.0,
        linestyle=(0, (4, 2)),
        label='Benchmark',
    )
    growth_ax.set_title(f'Growth of $1 ({covariance_method})', fontsize=16, fontweight='bold', pad=12)
    growth_ax.set_ylabel('Portfolio Value', fontsize=11)
    growth_ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'${y:,.2f}'))
    growth_ax.xaxis.set_major_locator(mdates.YearLocator(2))
    growth_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    growth_ax.set_xlim(series['Date'].iloc[0], series['Date'].iloc[-1])
    growth_ax.grid(axis='y', color=grid_color, linestyle='--', linewidth=0.8, alpha=0.9)
    growth_ax.grid(axis='x', alpha=0)
    growth_ax.spines['top'].set_visible(False)
    growth_ax.spines['right'].set_visible(False)
    growth_ax.legend(loc='upper left', frameon=False, fontsize=10)
    growth_ax.margins(x=0.01, y=0.08)

    relative_range = series['Relative Performance'].max() - series['Relative Performance'].min()
    relative_pad = max(relative_range, 0.002) * 0.20

    relative_fig, relative_ax = plt.subplots(
        figsize=(9.5, 3.8),
        facecolor='white',
        constrained_layout=True,
    )
    relative_ax.axhline(0.0, color=benchmark_color, linewidth=1.0, linestyle='--', alpha=0.9)
    relative_ax.fill_between(
        series['Date'],
        0.0,
        series['Relative Performance'],
        color=relative_color,
        alpha=0.18,
    )
    relative_ax.plot(
        series['Date'],
        series['Relative Performance'],
        color=relative_color,
        linewidth=2.2,
    )
    relative_ax.set_title('Relative Outperformance vs Benchmark', fontsize=15, fontweight='bold', pad=12)
    relative_ax.set_ylabel('Relative Return', fontsize=11)
    relative_ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.2%}'))
    relative_ax.xaxis.set_major_locator(mdates.YearLocator(2))
    relative_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    relative_ax.set_xlim(series['Date'].iloc[0], series['Date'].iloc[-1])
    relative_ax.set_ylim(
        series['Relative Performance'].min() - relative_pad,
        series['Relative Performance'].max() + relative_pad,
    )
    relative_ax.grid(axis='y', color=grid_color, linestyle='--', linewidth=0.8, alpha=0.9)
    relative_ax.grid(axis='x', alpha=0)
    relative_ax.spines['top'].set_visible(False)
    relative_ax.spines['right'].set_visible(False)
    relative_ax.annotate(
        f"End: {series['Relative Performance'].iloc[-1]:.2%}",
        xy=(series['Date'].iloc[-1], series['Relative Performance'].iloc[-1]),
        xytext=(-10, 0),
        textcoords='offset points',
        ha='right',
        va='center',
        fontsize=9,
        color=relative_color,
    )

    table_fig, table_ax = plt.subplots(
        figsize=(8.0, 3.6),
        facecolor='white',
        constrained_layout=True,
    )
    table_ax.axis('off')
    table_ax.set_title('Performance Summary', fontsize=14, fontweight='bold', pad=12)
    table = table_ax.table(
        cellText=summary_rows,
        colLabels=['Metric', 'Value'],
        cellLoc='left',
        colLoc='left',
        loc='center',
        colWidths=[0.72, 0.28],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(grid_color)
        if row == 0:
            cell.set_facecolor('#f4f4f4')
            cell.get_text().set_fontweight('bold')
        elif col == 1:
            cell.get_text().set_ha('right')

    if show:
        plt.show()

    return summary_df, growth_fig, relative_fig, table_fig


summary_df, growth_fig, relative_fig, table_fig = make_performance_outputs(
    backtest_results,
    COVARIANCE_METHOD,
    show=True,
)

In [ ]:
# Required output export for both covariance methods.
SAVE_RESULTS = True
RESULT_NAMES_BY_METHOD = {
    'sample_covariance_no_shrinkage': 'bond_em_no_shrinkage',
    'sklearn_ledoit_wolf': 'bond_em_ledoit_wolf',
}

reload(code_for_project_1)
save_experiment_outputs = code_for_project_1.save_experiment_outputs


def build_parameter_table(result_name, covariance_method, cov_shrinkage_by_date):
    parameter_rows = [
        {
            'Parameter': 'result_name',
            'Value': result_name,
            'Description': 'Folder name used for this saved output run.',
        },
        {
            'Parameter': 'asset_count',
            'Value': len(COMMON_ASSETS),
            'Description': 'Number of assets in the aligned investable universe.',
        },
        {
            'Parameter': 'risk_aversion_lambda',
            'Value': lambda_,
            'Description': 'Used consistently for implied returns and optimization.',
        },
        {
            'Parameter': 'black_litterman_tau',
            'Value': tau,
            'Description': 'Conservative BL prior uncertainty setting.',
        },
        {
            'Parameter': 'covariance_method',
            'Value': covariance_method,
            'Description': 'Covariance estimator used for this output run.',
        },
        {
            'Parameter': 'covariance_window_months',
            'Value': COVARIANCE_WINDOW,
            'Description': 'Trailing monthly returns used for annual covariance estimation.',
        },
        {
            'Parameter': 'lookback_signal_months',
            'Value': LOOKBACK_SIGNAL_MONTHS,
            'Description': 'Trailing months used to adapt weak view strength.',
        },
        {
            'Parameter': 'active_weight_bound_each_asset',
            'Value': active_bound,
            'Description': 'Maximum active deviation from benchmark for each asset; 2% is stricter than the 5% assignment limit.',
        },
        {
            'Parameter': 'rebalance_frequency',
            'Value': 'Annual target weights; monthly realized backtest returns',
            'Description': 'Portfolio weights are formed annually and applied to subsequent monthly returns.',
        },
        {
            'Parameter': 'final_year_diagnostics_date',
            'Value': valid_formation_dates[-1],
            'Description': 'Formation date used for final covariance, volatility, and correlation diagnostics.',
        },
    ]

    if not cov_shrinkage_by_date.empty:
        parameter_rows.extend([
            {
                'Parameter': 'ledoit_wolf_mean_shrinkage',
                'Value': cov_shrinkage_by_date.mean(),
                'Description': 'Average shrinkage intensity estimated by sklearn LedoitWolf across annual formation dates.',
            },
            {
                'Parameter': 'ledoit_wolf_min_shrinkage',
                'Value': cov_shrinkage_by_date.min(),
                'Description': 'Minimum shrinkage intensity estimated by sklearn LedoitWolf across annual formation dates.',
            },
            {
                'Parameter': 'ledoit_wolf_max_shrinkage',
                'Value': cov_shrinkage_by_date.max(),
                'Description': 'Maximum shrinkage intensity estimated by sklearn LedoitWolf across annual formation dates.',
            },
        ])

    return pd.DataFrame(parameter_rows)


def build_export_inputs(method, result, run_name):
    cov_matrices_ann = result['cov_matrices_ann']
    final_year_date = valid_formation_dates[-1]
    final_year_covariance = cov_matrices_ann[final_year_date].loc[
        COMMON_ASSETS, COMMON_ASSETS
    ].copy()
    final_year_covariance.index.name = 'Asset'

    final_year_volatility_series = pd.Series(
        np.sqrt(np.diag(final_year_covariance.to_numpy())),
        index=COMMON_ASSETS,
        name='Annualized Volatility',
    )
    final_year_volatility = final_year_volatility_series.reset_index()
    final_year_volatility.columns = ['Asset', 'Annualized Volatility']
    final_year_volatility.insert(0, 'Date', final_year_date)

    final_year_correlation = final_year_covariance.div(
        final_year_volatility_series,
        axis=0,
    ).div(
        final_year_volatility_series,
        axis=1,
    )
    final_year_correlation.index.name = 'Asset'

    annual_implied_returns = pd.DataFrame(result['implied_returns']).T
    annual_implied_returns.index.name = 'Date'
    annual_implied_returns = annual_implied_returns.reindex(columns=COMMON_ASSETS)

    annual_bl_posterior_returns = result['bl_posterior_table'].copy().reindex(columns=COMMON_ASSETS)
    annual_bl_posterior_returns.index.name = 'Date'

    annual_bl_shift_from_prior = result['bl_shift_table'].copy().reindex(columns=COMMON_ASSETS)
    annual_bl_shift_from_prior.index.name = 'Date'

    summary_df, growth_fig, relative_fig, table_fig = make_performance_outputs(
        result['backtest_results'],
        method,
        show=False,
    )

    parameter_table = build_parameter_table(
        result_name=run_name,
        covariance_method=method,
        cov_shrinkage_by_date=result['cov_shrinkage_by_date'],
    )

    return {
        'benchmark_weights': benchmarks,
        'view_specs': view_specs,
        'view_portfolio_weights': P,
        'backtest_results': result['backtest_results'],
        'optimal_weights': result['optimal_weights'],
        'active_weights': result['active_weights_table'],
        'optimization_diagnostics': result['optimization_diagnostics'],
        'holding_period_summary': result['holding_period_summary'],
        'adaptive_view_summary': result['adaptive_view_summary'],
        'q_table': result['Q_table'],
        'omega_multiplier_table': result['Omega_multiplier_table'],
        'parameter_table': parameter_table,
        'implied_returns': annual_implied_returns,
        'bl_posterior_returns': annual_bl_posterior_returns,
        'bl_shift_from_prior': annual_bl_shift_from_prior,
        'final_year_covariance': final_year_covariance,
        'final_year_volatility': final_year_volatility,
        'final_year_correlation': final_year_correlation,
        'summary_df': summary_df,
        'growth_fig': growth_fig,
        'relative_fig': relative_fig,
        'table_fig': table_fig,
    }


saved_paths_by_method = {}

if SAVE_RESULTS:
    for method, result in strategy_results.items():
        run_name = RESULT_NAMES_BY_METHOD[method]
        export_inputs = build_export_inputs(method, result, run_name)
        saved_paths = save_experiment_outputs(
            enabled=True,
            run_name=run_name,
            **export_inputs,
        )
        saved_paths_by_method[method] = saved_paths.assign(
            Method=method,
            Run_Name=run_name,
        )
        for figure_name in ['growth_fig', 'relative_fig', 'table_fig']:
            plt.close(export_inputs[figure_name])

    saved_paths_df = pd.concat(saved_paths_by_method.values(), ignore_index=True)
    display(saved_paths_df)
else:
    saved_paths_df = None
    print('SAVE_RESULTS is False, so no files were written.')